<h1 align = "center">
Rwanda Maize-Urea Trial
</h1>

<h2 align = "center">
Treatment Effect Analysis
</h2> 

# Objective

Our main objective here is to determine whether differences in maize yield across the four Urea treatments remain meaningful after accounting for season, mega-environment, and the fact that plots from the same field share growing conditions.

- We now have enough descriptive analysis from our EDA Notebook that we think motivates the next stage, formal treatment-effect analysis. 

- Here, we want to estimate treatment effects while accounting for the experimental design. 

    - Since the four treatments were assigned to plots within fields, we want to account for the shared field context rather than treating all 1,598 plots as completely independent observations.

- We want to begin with a treatment-effect model that includes treatment, season, mega-environment, and their relevant interactions, while accounting for field-level grouping. 

    - But before fitting it, we want to check that the field identifiers and treatment allocation can be represented appropriately in the model.

        - This should help us answer the more important question: *Are the observed yield differences sufficiently strong and consistent to support conclusions about treatment performance, once field and environmental variation are considered?*

## Baseline Treatment-Effect Model

Without complicating teh analysis upfront, we want to first fit the baseline treatment-effect model

**Why use a mixed-effects model?**

Our plots are grouped within fields, and each field was intended to contain plots receiving different treatments. 

- Plots from the same field are therefore not fully independent.

A linear mixed-effects model lets us estimate treatment differences while accounting for shared field-level conditions. We'll include:

- **Treatment** - *the main factor of interest.*

- **Season** - *to account for differences between 21A, 22A, and 22B.*

- **Mega-environment** - *to account for broad growing-environment differences.*

- **Field-season** - *as a random intercept to account for plots sharing the same experimental field in a season.*

This first model estimates adjusted treatment differences. It does not, by itself, establish that treatment assignment was randomized or prove causality.

### Imports and Setup

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

# Define path to folder
data_path = Path(r"../01_Data/processed")

def load_csv(filename):
    return pd.read_csv(data_path/filename)

In [3]:
copy_df = load_csv("plot_with_field_soil.csv")
copy_df.head()

,block_id,season,plot_id,treat_code,treat_details,urea_rate_timing,crop,prior_season_crop,planting_date,compost_tha,...,check_planting_date_before_harvest_date,days_silking_to_R3,calculated_urea_total,urea_total_difference,meg_env,_merge,pH,soil_match,has_soil_record,has_field_match
0,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-A,A,DAP 1Kg/Are + Urea 0.5Kg/Are at V6,Urea 0.5Kg/Are(V6),maize,maize,9/26/2020,8,...,False,20.0,50,0,Low-Mid-Dry,both,5.21,both,True,True
1,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-B,B,DAP 1Kg/Are + Urea 0.5Kg/Are at Planting and U...,Urea1kg/Are(Planting+V6),maize,maize,9/26/2020,8,...,False,20.0,100,0,Low-Mid-Dry,both,NaN,left_only,False,True
2,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-C,C,DAP 1Kg/Are + Urea 0.5Kg/Are at V6 + Urea 0.5K...,Urea 1kg/Are(V6 + V10),maize,maize,9/26/2020,8,...,False,20.0,100,0,Low-Mid-Dry,both,NaN,left_only,False,True
3,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-D,D,DAP 1Kg/Are at planting+ Urea 1Kg/Are at V6,Urea 1kg/Are(V6),maize,maize,9/26/2020,8,...,False,20.0,100,0,Low-Mid-Dry,both,NaN,left_only,False,True
4,d8807f25-e675-49ae-87a3-d228eca23c72,21A,d8807f25-e675-49ae-87a3-d228eca23c72-21A-A,A,DAP 1Kg/Are + Urea 0.5Kg/Are at V6,Urea 0.5Kg/Are(V6),maize,maize,9/26/2020,10,...,False,20.0,50,0,Low-Mid-Dry,both,5.75,both,True,True


Moveing forward with the analysis, we want to create a separate analysis copy to our plot_with_field_soil.csv so the original remains unchanged

In [4]:
import statsmodels.formula.api as smf

# Keep plots with the field context needed for this model.
# Five plots lack a matching mega-environment.
analysis_df = copy_df.dropna(
    subset=["meg_env"]
).copy()

# Identify each field within its season.
analysis_df["field_season_id"] = (
    analysis_df["block_id"].astype(str)
    + "_"
    + analysis_df["season"].astype(str)
)

# Set the 50 kg/ha treatment as the reference category.
treatment_order = [
    "Urea 0.5Kg/Are(V6)",
    "Urea1kg/Are(Planting+V6)",
    "Urea 1kg/Are(V6)",
    "Urea 1kg/Are(V6 + V10)"
]

analysis_df["urea_rate_timing"] = pd.Categorical(
    analysis_df["urea_rate_timing"],
    categories=treatment_order
)

# Fit the baseline mixed-effects model.
model = smf.mixedlm(
    "yield_tha ~ C(urea_rate_timing) + C(season) + C(meg_env)",
    data=analysis_df,
    groups=analysis_df["field_season_id"]
)

model_result = model.fit(reml=True)

print(model_result.summary())

                          Mixed Linear Model Regression Results
Model:                        MixedLM            Dependent Variable:            yield_tha 
No. Observations:             1593               Method:                        REML      
No. Groups:                   399                Scale:                         2.0696    
Min. group size:              3                  Log-Likelihood:                -3163.7807
Max. group size:              4                  Converged:                     Yes       
Mean group size:              4.0                                                         
------------------------------------------------------------------------------------------
                                                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------------------------
Intercept                                        3.690    0.198 18.601 0.000  3.301  4.079
C(urea_rate_timing)[T.Urea

### **What we can draw from the baseline model**

Our model converges successfully and uses 1,593 plots across 399 field-season groups. 

- This is a good starting point for estimating treatment differences while accounting for shared field conditions.

The main finding:

- Both 100 kg/ha treatments applied entirely or partly later in the season show higher estimated yields than the 50 kg/ha treatment, after adjusting for season and mega-environment.

- Adjusted treatment comparisons:

| Treatment comparison | Estimated yield difference (t/ha) | 95% confidence interval | p-value |
|---|---:|---:|---:|
| 100 kg/ha, split between planting and V6 | +0.144 | −0.056 to +0.344 | 0.158 |
| 100 kg/ha, entirely at V6 | +0.546 | +0.346 to +0.745 |

The reference treatment is 50 kg/ha applied entirely at V6.